# Homework 1: scraping and crawling

Information Retrieval, Innopolis University.

You write a small crawler and the pieces around it: a polite fetcher with a
cache, a page parser, a bit of text processing, the crawler itself, and a
scraper for a shop that hides its data behind a JSON API.

Everything runs against a practice website that starts on your own machine, so
nothing will depend on a real site, on your internet connection or on
somebody's rate limit.

## Rules

- Rename this file to your university email plus `.ipynb`, for example
  `i.ivanov@innopolis.university.ipynb`. A wrong file name means 0 points.
- Submit the notebook file only. Keep `ir_sandbox.py` next to it while you work,
  but do not submit it and do not change it.
- The grader imports your notebook and calls your functions. Only imports,
  function definitions, class definitions and simple constant assignments
  survive that import. Code you run at the top level of a cell (calls, prints,
  experiments) is ignored, so never define anything important there.
- Keep the names and the signatures exactly as given. A renamed function is a
  missing function.
- Allowed libraries: the Python standard library, `requests` and
  `beautifulsoup4`. No `lxml`, no `html5lib`, no `nltk`, no Selenium, no
  `aiohttp`, no `!pip install`. Use `BeautifulSoup(html, "html.parser")`.
- Python 3.10 or newer.

## Points

| Task | What | Points |
| --- | --- | --- |
| 1 | `PoliteFetcher`: cache, user agent, delays, timeouts, streamed download | 20 |
| 2 | `normalize_url` and `parse_page` | 20 |
| 3 | sentences, terms, ranking, two metrics | 15 |
| 4 | `Crawler` | 30 |
| 5 | shop behind a JSON API | 15 |

Partial credit is provided, so do try not to leave empty cells.
## Interview

After the deadline every student meets a TA for a few minutes and answers a
question or two about this code: why a function does what it does, what it
returns for an input you have not seen, what happens if the site answers
differently. The point is that you can explain and change your own work. You
may use any tool you like while writing it, including AI assistants, as long as
you understand every line you hand in.

## Self-check

`ir_sandbox.check(n, globals())` runs the same tests the grader runs, and the
final cell of this notebook grades the saved file the same way. We run them on
another copy of the site: same kind of pages, same rules, other addresses, other
texts and other products. So write code that follows the task, not code that
matches the pages you see here. Anything that hardcodes a path, a title or a
product of the practice site passes the self-check and fails the real run.


## The practice website

Run the next cell. It starts two small websites on your own machine:

- the museum site, the one you crawl
- a partner archive on another port, which has its own `robots.txt` with a
  `Crawl-delay`

It also prints an address where nothing is listening, which is handy for
testing connection errors, and a few test addresses for slow answers, error
codes and large downloads.

You can open the museum address in a browser and click around. Reading the page
source is part of the work. `SITE.requests()` gives you every request the site
has received, which is the easiest way to see what your code is really doing.


In [ ]:
import ir_sandbox

SITE = ir_sandbox.start()


The next cell is given. Put your email in `USER_AGENT`, that is the only change you need here.

In [ ]:
# Put your own email in USER_AGENT. Everything else in this cell is given.
import hashlib
import json
import math
import os
import re
import time
from collections import Counter, deque
from typing import Any, Dict, Iterator, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlsplit, urlunsplit
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup

USER_AGENT = "IRHomeworkBot/1.0 (+your.name@innopolis.university)"

# Largest piece of a download that may sit in memory at once.
CHUNK_SIZE = 256 * 1024

# The crawler never follows links whose path ends with one of these.
SKIP_EXTENSIONS = (".pdf", ".png", ".jpg", ".jpeg", ".gif", ".svg", ".zip", ".mp3", ".mp4")

# Words that carry no meaning for ranking (Task 3).
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "did", "do", "does",
    "for", "from", "has", "have", "how", "i", "in", "is", "it", "its", "many",
    "much", "my", "of", "on", "or", "that", "the", "their", "this", "to", "was",
    "were", "what", "when", "where", "which", "who", "why", "will", "with",
    "you", "your",
}

# A single dot after one of these words does not end a sentence (Task 3).
ABBREVIATIONS = {"dr", "mr", "mrs", "ms", "prof", "st", "mt", "vs", "no", "fig", "approx", "e.g", "i.e"}


## Task 1: `PoliteFetcher` (20 points)

A fetcher that is safe to point at a real website: it identifies itself, waits
between requests, times out, keeps a copy of what it downloads, and never
crashes on a broken server.

```python
class PoliteFetcher:
    def __init__(self, cache_dir="cache", user_agent=USER_AGENT, delay=0.0, timeout=5.0)
    def cache_paths(self, url) -> tuple[str, str]
    def set_host_delay(self, netloc, seconds) -> None
    def fetch(self, url) -> dict
    def download(self, url, path) -> str | None
```

`fetch(url)` returns a dict with exactly these keys:

- `url`: the URL you were asked to fetch, unchanged
- `final_url`: the URL after redirects, or `url` when the request failed
- `status`: the HTTP status code, or `None` when there was no answer at all
  (timeout, connection error, too many redirects)
- `content_type`: the part of the `Content-Type` header before the first
  semicolon, lowercase and stripped, `""` when there was no answer
- `body`: the response bytes when the status is 200, otherwise `None`
- `from_cache`: `True` when the answer came from disk

Rules:

* Only responses with status 200 should be saved in the cache. Store the body in one file and the other response fields in a small JSON file next to it. `cache_paths(url)` should return these two paths, and both files must be inside `cache_dir`. URLs that differ in any way, including the query string, must use different files.

* If a new `PoliteFetcher` is created with the same `cache_dir`, it should be able to return a cached response from disk without making a network request. It should return the same status, content type, final URL, and body as the original response.

* Every request should use the user agent given to the constructor. Store it in `self.user_agent`, since the crawler in Task 4 reads it from there.

* Wait at least `delay` seconds between requests to the same host. `set_host_delay(netloc, seconds)` should increase the delay for one specific host. The `netloc` looks like `127.0.0.1:8765`. Delays are tracked separately for each host, and cache hits should never cause a wait.

* Pass `timeout` to `requests`. Also, `fetch` must never raise an exception.

* `download(url, path)` should stream the response body to disk using chunks no larger than `CHUNK_SIZE` bytes and return the SHA-256 hex digest of the downloaded data. The entire file must never be kept in memory. The tests download 32 MiB and use `tracemalloc` to check memory usage.

* `download(url, path)` should return `None` and leave no file behind when the response status is not 200, when the number of received bytes does not match `Content-Length`, or when the server sends an `X-Checksum-SHA256` header and its value does not match the SHA-256 digest of the data received.

* Write the downloaded data to a temporary file first. Rename it to `path` only after the download finishes successfully. This makes sure that a failed download cannot leave a partial file at `path`.



In [ ]:
class PoliteFetcher:
    """Downloads pages politely and keeps successful responses in a disk cache."""

    def __init__(self, cache_dir: str = "cache", user_agent: str = USER_AGENT,
                 delay: float = 0.0, timeout: float = 5.0) -> None:
        raise NotImplementedError

    def cache_paths(self, url: str) -> Tuple[str, str]:
        """Return (body_path, meta_path) for this URL, both inside cache_dir."""
        raise NotImplementedError

    def set_host_delay(self, netloc: str, seconds: float) -> None:
        """Use at least `seconds` between requests to this host."""
        raise NotImplementedError

    def fetch(self, url: str) -> Dict[str, Any]:
        """Return url, final_url, status, content_type, body and from_cache."""
        raise NotImplementedError

    def download(self, url: str, path: str) -> Optional[str]:
        """Stream a big file to `path`. Return its sha256 hex digest, or None."""
        raise NotImplementedError


In [ ]:
# Try your fetcher. This cell is not graded.
fetcher = PoliteFetcher(cache_dir="cache_demo", user_agent=USER_AGENT, delay=0.2)
first = fetcher.fetch(SITE.url("/about/"))
print(first["status"], first["content_type"], len(first["body"] or b""), first["from_cache"])
print("second time from cache:", fetcher.fetch(SITE.url("/about/"))["from_cache"])
print("missing page:", fetcher.fetch(SITE.url("/events/2019-gala"))["status"])
print("nothing listening:", fetcher.fetch(SITE.closed + "/")["status"])
print(SITE.requests()[-3:])


In [ ]:
ir_sandbox.check(1, globals())

## Task 2: URLs and page parsing (20 points)

```python
def normalize_url(url, base=None) -> str | None
def parse_page(html, page_url) -> dict
```

`normalize_url` takes whatever is in an `href` and turns it into one canonical form so the crawler can compare URLs.

* Strip spaces around the URL. When `base` is given, resolve the URL against it with `urljoin`.

* Convert the scheme and host to lowercase. Remove `user:password@`. Remove the port when it is the default one for the scheme, which is 80 for http and 443 for https. Keep any other port.

* Return `None` for anything that is not http or https, and for URLs that have no host.

* Remove the fragment.

* When the path is empty, use `/`. Resolve dot segments according to RFC 3986 section 5.2.4. For example, `/a/b/../c` becomes `/a/c`.

* Remove query parameters whose name starts with `utm_` in any case. Remove empty parameters. Keep all other parameters in their original order. When nothing is left, remove the query string completely.

* Never raise an exception. Return `None` when the URL cannot be parsed.

`parse_page(html, page_url)` should return a dict with the keys `title`, `links`, `images`, `paragraphs`, `text`, `canonical` and `meta_robots`.

* Parse the HTML with `BeautifulSoup(html, "html.parser")`.

* Before reading anything, remove every `script`, `style`, `noscript` and `template` element, as well as every element with a `hidden` attribute. One removed element may be inside another removed element, so handle nested removed elements correctly.

* The base URL is the `href` of the first `base` tag resolved against `page_url`. When there is no `base` tag, use `page_url`. Resolve every URL on the page against this base URL, including an empty `href`.

* `title` is the text inside `title` with whitespace collapsed. Use `""` when there is no title.

* `links` contains one dict for each `a` element that has an `href`, in document order. Each dict has `text`, `url` and `nofollow`. `text` is the anchor text with whitespace collapsed. `url` is the normalized URL. `nofollow` is `True` when the `rel` attribute contains `nofollow` in any case. Skip links when `normalize_url` returns `None`.

* `images` contains the normalized URLs from `img` elements. Use `data-src` when it exists and is not blank. Otherwise use `src`. Skip blank values and duplicates, but keep the original document order.

* `paragraphs` contains the text of every `p` element inside `main`. When there is no `main`, use `body` instead. Collapse whitespace and remove empty paragraphs.

* `text` contains all visible text from `body`, with all whitespace collapsed into single spaces.

* `canonical` is the normalized `href` from `link rel="canonical"`. The `rel` value can use any letter case. Use `None` when there is no canonical link.

* `meta_robots` is a set of lowercase directives from `meta name="robots"`, split by commas. Use an empty set when the tag is missing.

In [ ]:
def normalize_url(url: str, base: Optional[str] = None) -> Optional[str]:
    """Resolve and canonicalize a URL, or return None if it is not http(s)."""
    raise NotImplementedError


def parse_page(html: bytes, page_url: str) -> Dict[str, Any]:
    """Return title, links, images, paragraphs, text, canonical and meta_robots."""
    raise NotImplementedError


In [ ]:
# Try your parser. This cell is not graded.
page = PoliteFetcher(cache_dir="cache_demo", user_agent=USER_AGENT).fetch(SITE.url("/halls/rivers"))
parsed = parse_page(page["body"], SITE.url("/halls/rivers"))
print(parsed["title"], "|", parsed["canonical"], "|", sorted(parsed["meta_robots"]))
for link in parsed["links"][6:9]:
    print(link)
print(parsed["images"])
print(parsed["paragraphs"][0])


In [ ]:
ir_sandbox.check(2, globals())

## Task 3: sentences, terms and ranking (15 points)

```python
def split_sentences(text) -> list[str]
def tokenize(text) -> list[str]
def top_terms(texts, n=10) -> list[tuple[str, int]]
def rank_sentences(question, sentences) -> list[int]
def precision_at_k(ranked, relevant, k) -> float
def reciprocal_rank(ranked, relevant, k) -> float
def best_answer(question, pages) -> tuple[str, str] | None
```
`split_sentences` should split the text into sentences.

* A sentence ends at `.`, `!` or `?`. These characters can appear more than once.

* They can be followed by closing quotes or brackets, then whitespace, and then an uppercase letter, a digit, or an opening quote or bracket.

* A single `.` does not end a sentence when the word before it is in `ABBREVIATIONS` or when that word has only one letter. Compare the word in lowercase after removing any opening brackets and quotes from the beginning.

* Strip each sentence, remove empty ones, and do not change anything else inside the sentences.

`tokenize` should lowercase the text and return all runs of letters and digits using `re.findall(r"[^\W_]+", text.lower())`. Letters outside the Latin alphabet should also count as letters.

`top_terms` should count all tokens from all texts together.

* Remove tokens from `STOPWORDS`.
* Remove tokens that contain only digits.
* Sort first by count from highest to lowest, then alphabetically by the term.
* Return the first `n` pairs.

`rank_sentences` should rank sentences based on the question.

* The query terms are `tokenize(question)` with anything in `STOPWORDS` removed.

* A sentence gets a score by adding the score of every query term that appears in it. Each term counts only once per sentence.

* The score for each term is `log((1 + N) / (1 + df)) + 1`, where `N` is the total number of sentences and `df` is the number of sentences containing that term.

* Remove sentences with a score of 0.

* Return the indices of the remaining sentences, sorted by highest score first. When scores are tied, sort by fewer tokens first, then by the smaller index.

`precision_at_k` is the number of the first `k` items in `ranked` that are also in `relevant`, divided by `k`.

`reciprocal_rank` is `1` divided by the position of the first relevant item among the first `k` items in `ranked`, counting positions from 1. Return `0.0` when there is no relevant item.

`best_answer` takes `pages`, which is a list of dicts containing `url` and `paragraphs`.

* Split every paragraph into sentences.
* Rank all sentences together.
* Return the best matching sentence as `(sentence, url of its page)`.
* Return `None` when no sentence matches.

In [ ]:
def split_sentences(text: str) -> List[str]:
    raise NotImplementedError


def tokenize(text: str) -> List[str]:
    raise NotImplementedError


def top_terms(texts: List[str], n: int = 10) -> List[Tuple[str, int]]:
    raise NotImplementedError


def rank_sentences(question: str, sentences: List[str]) -> List[int]:
    """Indices of the matching sentences, best first."""
    raise NotImplementedError


def precision_at_k(ranked: List[int], relevant: Set[int], k: int) -> float:
    raise NotImplementedError


def reciprocal_rank(ranked: List[int], relevant: Set[int], k: int) -> float:
    raise NotImplementedError


def best_answer(question: str, pages: List[Dict[str, Any]]) -> Optional[Tuple[str, str]]:
    """Best sentence over all pages, as (sentence, page url)."""
    raise NotImplementedError


In [ ]:
# Try your text functions. This cell is not graded.
print(split_sentences("Dr. Petrova found it in 1978. The skeleton is 3.2 m tall! Where? Near Kazan."))
print(tokenize("The Mammoth's tusk: 2.4 m!"))
print(top_terms(["rivers and fish", "fish, fish and lakes"], 3))
print(rank_sentences("how tall is the skeleton", ["The skeleton is 3.2 m tall.", "Tickets cost 150 rubles."]))
print(precision_at_k([3, 1, 2], {1, 2}, 2), reciprocal_rank([4, 2, 7], {7}, 3))


In [ ]:
ir_sandbox.check(3, globals())

## Task 4: the crawler (30 points)

```python
class Crawler:
    def __init__(self, start_url, fetcher, max_pages=20, max_depth=2, allowed_hosts=None)
    def crawl(self)          # generator
    self.errors              # list of (url, status)
    self.blocked             # list of URLs robots.txt kept you away from
```
Your crawler may only use `fetcher.fetch(url)`, `fetcher.set_host_delay(netloc, seconds)` and `fetcher.user_agent` from the fetcher. You may also use your own `normalize_url` and `parse_page`. We test this task with our own fetcher and parser, so bugs in Task 1 or Task 2 will not be counted against you again here.

* Normalize `start_url`. When `allowed_hosts` is `None`, only the host of the start URL is allowed. A host is written as a netloc such as `127.0.0.1:8765`.

* Crawl in breadth-first order using a FIFO queue. The start page has depth 0.

* Before making the first request to a host, fetch that host's `/robots.txt` through the fetcher. Read it with `urllib.robotparser.RobotFileParser().parse(text.splitlines())`. Do this only once for each host. If `robots.txt` is missing or does not return status 200, allow everything on that host. When it defines a `Crawl-delay` for your user agent, pass that delay to `fetcher.set_host_delay`.

* When `robots.txt` forbids a URL, add it to `self.blocked` and never request it.

* Each page request counts as one attempt. Requests for `robots.txt` do not count. Stop when you have made `max_pages` page attempts or when the queue is empty.

* When a page returns a status other than 200, add `(url, status)` to `self.errors` and do not yield anything for that page.

* When the final URL after redirects is different from the requested URL, skip the page when the final URL is outside `allowed_hosts` or has already been seen. Otherwise, mark the final URL as seen and use the final URL in the returned record.

* Only parse pages whose content type is `text/html`.

* Yield `{"url": final url, "depth": depth, "title": title, "links": [link urls in document order]}` unless the page has `noindex` in its meta robots.

* Do not follow any links when the page has `nofollow` in its meta robots, or when its depth is already `max_depth`.

* Skip links when they have `rel="nofollow"`, have already been seen, are outside `allowed_hosts`, or have a path ending with one of the extensions in `SKIP_EXTENSIONS`.

* `crawl()` must be a generator. It should fetch the next page only when the caller asks for the next record. For example, `itertools.islice(crawler.crawl(), 2)` should make only two page requests, not twenty.

The practice site is designed to test these cases: a redirect, a redirect loop, a 404, a 500, a page with `noindex`, a page with `nofollow`, a `base` tag, an endless calendar, a PDF, a JSON endpoint and a partner host with a crawl delay.


In [ ]:
class Crawler:
    """Breadth-first crawler that yields one record per indexable HTML page."""

    def __init__(self, start_url: str, fetcher: "PoliteFetcher", max_pages: int = 20,
                 max_depth: int = 2, allowed_hosts: Optional[Set[str]] = None) -> None:
        self.errors: List[Tuple[str, Optional[int]]] = []
        self.blocked: List[str] = []
        raise NotImplementedError

    def can_fetch(self, url: str) -> bool:
        """True when robots.txt of that host allows this URL."""
        raise NotImplementedError

    def crawl(self) -> Iterator[Dict[str, Any]]:
        """Yield {"url", "depth", "title", "links"} for every indexable page."""
        raise NotImplementedError


In [ ]:
# Try your crawler. This cell is not graded.
fetcher = PoliteFetcher(cache_dir="cache_crawl", user_agent=USER_AGENT)
crawler = Crawler(SITE.url("/"), fetcher, max_pages=8, max_depth=2)
for record in crawler.crawl():
    print(record["depth"], record["url"], "|", record["title"], "|", len(record["links"]), "links")
print("blocked by robots.txt:", crawler.blocked)
print("errors:", crawler.errors)


In [ ]:
ir_sandbox.check(4, globals())

## Task 5: the shop behind a JSON API (15 points)

The shop page shows nothing useful in its HTML: the products arrive later from
a JSON API. Open `SITE.url("/shop/")` in a browser, look at the page source,
and you will find the configuration the page itself uses.

```python
def parse_price(value) -> float | None
def normalize_product(raw) -> dict
def scrape_shop(shop_url, user_agent=USER_AGENT) -> list[dict]
def filter_products(products, tag=None, max_price=None, min_rating=None, in_stock_only=False) -> list[int]
```

`parse_price` should handle the values that the API actually sends.

* When the value is an `int` or `float`, return it as a float rounded to two decimal places. `True` does not count as a number.
* When the value is a string, keep only digits, commas and dots and remove everything else.
* When both a comma and a dot are present, treat the comma as a thousands separator.
* When there are only commas, treat a single comma followed by exactly two digits as a decimal comma. Otherwise, treat the commas as thousands separators.
* Return `None` when there is nothing sensible left after cleaning the value.

`normalize_product(raw)` should return:

* `id`: an `int`
* `name`: the title with whitespace collapsed
* `price`: the result of `parse_price` on the price field
* `rating`: a float out of 5. For example, `"4,5"` becomes 4.5 and `"9/10"` becomes 4.5. An empty or missing rating becomes `None`.
* `tags`: lowercase and stripped, with empty strings and duplicates removed. Keep the original order. When tags are given as a string, split them on commas.
* `in_stock`: `True` for `True`, for numbers greater than zero, and for the strings `yes`, `in stock` and `available` in any case. Otherwise return `False`.

`scrape_shop` should load the shop page, get the endpoint, header name and token from the configuration, and then walk through the API.

* Send the token in the header whose name is given in the configuration on every API request.
* The token changes every time the shop page is loaded, so use the token from the page you just loaded.
* Follow `next` until it becomes `null`. Resolve each `next` URL against the URL of the response where that `next` value was found.
* When the API returns `429` with a `Retry-After` header, wait for that many seconds and then repeat the same request.
* A product may appear on more than one page. Keep the first copy you see and preserve the order given by the API.
* Return `normalize_product` for every product.

`filter_products` should keep products that match every filter that is not `None`. The filters are a tag compared in lowercase, a maximum price, a minimum rating and an in-stock requirement.

* A product without a price does not pass a price filter.
* A product without a rating does not pass a rating filter.
* Sort first by rating from highest to lowest, with missing ratings last.
* Then sort by price from lowest to highest, with missing prices last.
* Then sort by `id`.
* Return the list of product ids.



In [ ]:
def parse_price(value: Any) -> Optional[float]:
    raise NotImplementedError


def parse_rating(value: Any) -> Optional[float]:
    """Helper, used by normalize_product. The tests do not call it directly."""
    raise NotImplementedError


def parse_stock(value: Any) -> bool:
    """Helper, used by normalize_product. The tests do not call it directly."""
    raise NotImplementedError


def normalize_product(raw: Dict[str, Any]) -> Dict[str, Any]:
    raise NotImplementedError


def scrape_shop(shop_url: str, user_agent: str = USER_AGENT) -> List[Dict[str, Any]]:
    raise NotImplementedError


def filter_products(products: List[Dict[str, Any]], tag: Optional[str] = None,
                    max_price: Optional[float] = None, min_rating: Optional[float] = None,
                    in_stock_only: bool = False) -> List[int]:
    raise NotImplementedError


In [ ]:
# Try your shop scraper. This cell is not graded.
products = scrape_shop(SITE.url("/shop/"), USER_AGENT)
print(len(products), "products")
for product in products[:3]:
    print(product)
print("cheap and in stock:", filter_products(products, max_price=1000, in_stock_only=True))


In [ ]:
ir_sandbox.check(5, globals())

## Before you submit

1. Restart the kernel and run the whole notebook from the top. The grader
   imports a fresh copy of your code, so anything that only works because of a
   variable you created by hand will fail there.
2. Save the notebook under your email name, for example
   `i.ivanov@innopolis.university.ipynb`.
3. Run the cell below with that file name. It extracts your code exactly the way
   the grader does and runs every test.
4. Upload the notebook to Moodle. Nothing else.


In [ ]:
# Change the file name to your own, save the notebook, then run this cell.
ir_sandbox.check_notebook("your.name@innopolis.university.ipynb")
